<a href="https://colab.research.google.com/github/BWMIN-Hub/SR_practice/blob/main/notebooks/01_srcnn_x3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SRCNN ×3 — The Beginning of Deep Learning-based Super-Resolution

흐린 위성사진(10 m) → 3배 선명하게(3.33 m).


## 1. 데이터

In [ ]:
import sys, urllib.request

LIB = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main/lib'
for m in ['sr_utils.py', 'srcnn_arch.py', 'srcnn_imgproc.py', 'srcnn_models.py']:
    urllib.request.urlretrieve(f'{LIB}/{m}', m)
    sys.modules.pop(m[:-3], None)     # 이미 불러온 옛 모듈이 남아 있으면 비운다

from sr_utils import *

show_data()        # validation 2패치 + test 2구역

## 2. 훈련

**코드가 도는지 확인하는 용도다.** 16장으로 1 epoch 만 돌린다.
아래 결과는 전체 데이터로 학습해둔 가중치를 쓴다.

모델이 워낙 작아 GPU 가 없어도 돈다. `srcnn_pairs` 가 LR 을 bicubic 으로 3배 키운 뒤
Y 채널만 뽑아 입력을 만들고, 목표는 HR 의 Y 채널이다.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from srcnn_models import build_srcnn, srcnn_pairs

N_TRAIN, EPOCHS, BATCH = 16, 1, 4
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

lo, hi = zip(*[pair('training', s) for s in list_split('training')[:N_TRAIN]])
x, t = srcnn_pairs(lo, hi)        # 입력: bicubic 으로 키운 LR 의 Y / 목표: HR 의 Y
loader = DataLoader(TensorDataset(x, t), batch_size=BATCH, shuffle=True)

net = build_srcnn().to(dev).train()
opt = torch.optim.Adam(net.parameters(), 2e-4, betas=(0.9, 0.99))
crit = nn.MSELoss()

print(f'SRCNN  {sum(p.numel() for p in net.parameters())/1e3:.1f}K')
for ep in range(1, EPOCHS + 1):
    tot = 0.0
    for xb, tb in loader:
        loss = crit(net(xb.to(dev)), tb.to(dev))
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item()
    print(f'epoch {ep}/{EPOCHS}   MSE {tot/len(loader):.6f}')

## 3. 결과

In [ ]:
MODEL = f'{BASE}/models/01_srcnn_x3'

from srcnn_models import load_srcnn, srcnn_upscale

net = load_srcnn(fetch(f'{MODEL}/checkpoints/srcnn_x3.pth', 'srcnn_x3.pth'))
print(f'SRCNN  {sum(p.numel() for p in net.parameters())/1e3:.1f}K')

upscale = lambda lr: srcnn_upscale(net, lr)

show_results(upscale, 'SRCNN', center=[(67, 370), (313, 36)])

## 4. 최종 테스트

정답이 없는 실제 Sentinel-2 촬영본이다. 눈으로 확인한다.


In [ ]:
show_test(upscale, 'SRCNN', center=(800, 800), size=70)